# Loan Status – Logistic Regression Projesi
Bu notebook, veri keşfi (EDA), ön işleme, modelleme ve değerlendirme adımlarını içerir.

## 1. Gerekli Kütüphaneler

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    mean_squared_error,
    r2_score,
    roc_auc_score,
    roc_curve
)


## 2. Veriyi Okuma

In [ ]:

df = pd.read_csv("C:/Users/ASUS/Desktop/veri/veriseti/logistic_regression.csv")
df.head()


## 3. EDA (Veri Keşfi)

In [ ]:

df.info()


In [ ]:

df.describe()


In [ ]:

print("Eksik Değerler:\n")
df.isnull().sum()


In [ ]:

sns.countplot(x="loan_status", data=df)
plt.title("Loan Status Dağılımı")
plt.show()


## 4. Ön İşleme

In [ ]:

cat_cols = df.select_dtypes(include="object").columns
num_cols = df.select_dtypes(exclude="object").columns

df[cat_cols] = SimpleImputer(strategy="most_frequent").fit_transform(df[cat_cols])
df[num_cols] = SimpleImputer(strategy="mean").fit_transform(df[num_cols])

le = LabelEncoder()
for col in cat_cols:
    df[col] = le.fit_transform(df[col])

df.head()


## 5. Bağımsız / Bağımlı Değişkenler

In [ ]:

X = df.drop("loan_status", axis=1)
y = df["loan_status"]


## 6. Ölçeklendirme

In [ ]:

scaler = StandardScaler()
X = scaler.fit_transform(X)


## 7. Train - Test Ayırma

In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


## 8. Model Kurulumu

In [ ]:

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)


## 9. Tahmin

In [ ]:

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]


## 10. Değerlendirme Metrikleri

In [ ]:

print("Accuracy:", accuracy_score(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))
print("R2 Score:", r2_score(y_test, y_pred))
print("ROC-AUC Score:", roc_auc_score(y_test, y_prob))

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

conf_matrix = confusion_matrix(y_test, y_pred)
conf_matrix


## 11. Görselleştirmeler

In [ ]:

sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Tahmin")
plt.ylabel("Gerçek")
plt.title("Confusion Matrix")
plt.show()


In [ ]:

fpr, tpr, _ = roc_curve(y_test, y_prob)
plt.plot(fpr, tpr, label=f"AUC = {roc_auc_score(y_test, y_prob):.2f}")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()
